Dataset preprocessing

In [1]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tools.preprocessor import NLProcessor
import pandas as pd
import numpy as np
import torch

df1 = pd.read_csv("data/intents_train.csv")
df2 = pd.read_csv("data/bike_shop_intents.csv")
df3 = pd.read_csv("data/bike_shop_intents2.csv")

df = pd.concat([df1, df2, df3], ignore_index=True)

print("data frame shape:", df.shape)

del df1, df2, df3

text_train, text_test, intent_train, intent_test = train_test_split(
    df['text'],
    df['intent'],
    test_size=0.2,
    stratify=df['intent'],
    random_state=42
)

processor = NLProcessor(use_stopwords=True)
le        = LabelEncoder()

X_train  = processor.transform(text_train).astype(np.float32)
y_train  = le.fit_transform(intent_train).astype(np.int64)

y_train  = np.delete(y_train, processor.unfound_words)
processor.unfound_words = []

X_train = torch.from_numpy(X_train)
y_train = torch.from_numpy(y_train)
classes_ = le.classes_

print("y classes :", le.classes_)
print("X train shape:", X_train.shape)
print("y train shape:", y_train.shape)

X_test = processor.transform(text_test).astype(np.float32)
y_test = le.transform(intent_test).astype(np.int64)
y_test = np.delete(y_test, processor.unfound_words)

X_test = torch.from_numpy(X_test)
y_test = torch.from_numpy(y_test)

print("X test shape:", X_test.shape)
print("y test shape:", y_test.shape)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32
)

data frame shape: (333, 2)
y classes : ['Bike Types' 'Cost Estimation' 'Hours' 'Make Appointment' 'Return Policy'
 'Trade-in Options' 'Welcome Intent']
X train shape: torch.Size([263, 7, 300])
y train shape: torch.Size([263])
X test shape: torch.Size([67, 7, 300])
y test shape: torch.Size([67])


RNN Solution

In [ ]:
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from tools.rnn import RNN
import torch.nn as nn


model = RNN(
    input_size=X_train.shape[2],
    hidden_size=128,
    num_layers=2,
    num_classes=len(classes_),
    dropout_rate=0.3
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
num_epochs = 48

for epoch in range(num_epochs):

    model.train()
    train_loss = 0
    train_correct = 0

    for batch_X, batch_y in train_loader:

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss    = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(dim=1) == batch_y).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc      = train_correct / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/20] Loss: {train_loss/len(train_loader):.4f}")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.3f}"
    )

preds = None
with torch.no_grad():

    outputs = model(X_test)
    preds   = torch.argmax(outputs, dim=1)

print("RNN accuracy score:",accuracy_score(
    y_true=y_test.numpy(),
    y_pred=preds.numpy()
    )
)

torch.save(model, "models/rnn.pkl")
print("RNN model saved as models/rnn.pkl")

Epoch [1/20] Loss: 1.9119
Epoch [1/48] Train Loss: 1.9119  Acc: 0.266
Epoch [2/20] Loss: 1.5037
Epoch [2/48] Train Loss: 1.5037  Acc: 0.354
Epoch [3/20] Loss: 1.2692
Epoch [3/48] Train Loss: 1.2692  Acc: 0.513
Epoch [4/20] Loss: 0.9554
Epoch [4/48] Train Loss: 0.9554  Acc: 0.654
Epoch [5/20] Loss: 0.6666
Epoch [5/48] Train Loss: 0.6666  Acc: 0.772
Epoch [6/20] Loss: 0.4846
Epoch [6/48] Train Loss: 0.4846  Acc: 0.844
Epoch [7/20] Loss: 0.3918
Epoch [7/48] Train Loss: 0.3918  Acc: 0.886
Epoch [8/20] Loss: 0.2566
Epoch [8/48] Train Loss: 0.2566  Acc: 0.939
Epoch [9/20] Loss: 0.2246
Epoch [9/48] Train Loss: 0.2246  Acc: 0.935
Epoch [10/20] Loss: 0.1913
Epoch [10/48] Train Loss: 0.1913  Acc: 0.943
Epoch [11/20] Loss: 0.2039
Epoch [11/48] Train Loss: 0.2039  Acc: 0.939
Epoch [12/20] Loss: 0.1273
Epoch [12/48] Train Loss: 0.1273  Acc: 0.970
Epoch [13/20] Loss: 0.2104
Epoch [13/48] Train Loss: 0.2104  Acc: 0.939
Epoch [14/20] Loss: 0.1459
Epoch [14/48] Train Loss: 0.1459  Acc: 0.958
Epoch [15/

LSTM Solution

In [ ]:
from tools.lstm import LSTM

model = LSTM(
    input_size=X_train.shape[2],
    hidden_size=128,
    num_layers=2,
    num_classes=len(classes_),
    dropout_rate=0.5
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
    
num_epochs   = 29
for epoch in range(num_epochs):

    model.train()
    train_loss = 0
    train_correct = 0

    for batch_X, batch_y in train_loader:

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss    = criterion(outputs, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(dim=1) == batch_y).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc      = train_correct / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/20] Loss: {train_loss/len(train_loader):.4f}")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.3f}"
    )
    
preds = None
with torch.no_grad():

    outputs = model(X_test)
    preds   = torch.argmax(outputs, dim=1)

print("LSTM accuracy score:",accuracy_score(
        y_true=y_test.numpy(),
         y_pred=preds.numpy()
    )
)

torch.save(model, "models/lstm.pkl")
print("LSTM model saved as models/lstm.pkl")

Epoch [1/20] Loss: 1.9368
Epoch [1/29] Train Loss: 1.9368  Acc: 0.183
Epoch [2/20] Loss: 1.7461
Epoch [2/29] Train Loss: 1.7461  Acc: 0.327
Epoch [3/20] Loss: 1.4193
Epoch [3/29] Train Loss: 1.4193  Acc: 0.426
Epoch [4/20] Loss: 1.1070
Epoch [4/29] Train Loss: 1.1070  Acc: 0.532
Epoch [5/20] Loss: 0.8474
Epoch [5/29] Train Loss: 0.8474  Acc: 0.662
Epoch [6/20] Loss: 0.6787
Epoch [6/29] Train Loss: 0.6787  Acc: 0.760
Epoch [7/20] Loss: 0.5080
Epoch [7/29] Train Loss: 0.5080  Acc: 0.798
Epoch [8/20] Loss: 0.4164
Epoch [8/29] Train Loss: 0.4164  Acc: 0.894
Epoch [9/20] Loss: 0.3574
Epoch [9/29] Train Loss: 0.3574  Acc: 0.894
Epoch [10/20] Loss: 0.2670
Epoch [10/29] Train Loss: 0.2670  Acc: 0.924
Epoch [11/20] Loss: 0.2000
Epoch [11/29] Train Loss: 0.2000  Acc: 0.939
Epoch [12/20] Loss: 0.1663
Epoch [12/29] Train Loss: 0.1663  Acc: 0.943
Epoch [13/20] Loss: 0.1804
Epoch [13/29] Train Loss: 0.1804  Acc: 0.951
Epoch [14/20] Loss: 0.1220
Epoch [14/29] Train Loss: 0.1220  Acc: 0.958
Epoch [15/

CNN Solution

In [ ]:
from tools.cnn import TextCNN

X_train = X_train.permute(0, 2, 1)
batch_size, embed_dim, seq_len = X_train.shape
dataloader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32
)
# ------------------------------------------------------------------
# Model, loss, optimiser, scheduler
# ------------------------------------------------------------------
model = TextCNN(
    embed_dim=embed_dim,
    out_channels=128,
    output_size=len(classes_),
    kernel_sizes=[3, 5, 7],
    dropout_rate=0.5,
)
 
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)  # mild label smoothing
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
 
# Reduce LR by ×0.5 whenever train loss stops improving for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

epochs = 30
for epoch in range(epochs):
    model.train()
    train_loss    = 0.0
    train_correct = 0
 
    for X_batch, labels in dataloader:
        optimizer.zero_grad()
 
        outputs = model(X_batch)
        loss    = criterion(outputs, labels)
 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
 
        train_loss    += loss.item()
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
 
    avg_loss = train_loss / len(dataloader)
    acc      = train_correct / len(dataloader.dataset)
    lr_now   = optimizer.param_groups[0]["lr"]
 
    print(
        f"Epoch [{epoch+1:>3}/{epochs}]  "
        f"Loss: {avg_loss:.4f}  Acc: {acc:.3f}  LR: {lr_now:.2e}"
    )
    scheduler.step(avg_loss)
 
X_test = X_test.permute(0, 2, 1)

preds = None
try:
    with torch.no_grad():
    
        outputs = model(X_test)
        preds   = torch.argmax(outputs, dim=1)
    
    print("Text CNN accuracy score:",accuracy_score(
            y_true=y_test.numpy(),
            y_pred=preds.numpy()
        )
    )
except Exception as e:
    print(str(e))
torch.save(model.state_dict(), "models/text_cnn.pkl")
print("CNN saved as models/text_cnn.pkl")

Epoch [  1/30]  Loss: 1.7078  Acc: 0.376  LR: 1.00e-03
Epoch [  2/30]  Loss: 0.9353  Acc: 0.722  LR: 1.00e-03
Epoch [  3/30]  Loss: 0.6498  Acc: 0.882  LR: 1.00e-03
Epoch [  4/30]  Loss: 0.4794  Acc: 0.943  LR: 1.00e-03
Epoch [  5/30]  Loss: 0.3983  Acc: 0.973  LR: 1.00e-03
Epoch [  6/30]  Loss: 0.3697  Acc: 0.992  LR: 1.00e-03
Epoch [  7/30]  Loss: 0.3531  Acc: 0.989  LR: 1.00e-03
Epoch [  8/30]  Loss: 0.3299  Acc: 1.000  LR: 1.00e-03
Epoch [  9/30]  Loss: 0.3382  Acc: 1.000  LR: 1.00e-03
Epoch [ 10/30]  Loss: 0.3145  Acc: 0.996  LR: 1.00e-03
Epoch [ 11/30]  Loss: 0.3114  Acc: 0.996  LR: 1.00e-03
Epoch [ 12/30]  Loss: 0.3188  Acc: 0.996  LR: 1.00e-03
Epoch [ 13/30]  Loss: 0.3223  Acc: 1.000  LR: 1.00e-03
Epoch [ 14/30]  Loss: 0.3117  Acc: 0.996  LR: 1.00e-03
Epoch [ 15/30]  Loss: 0.3050  Acc: 1.000  LR: 1.00e-03
Epoch [ 16/30]  Loss: 0.3015  Acc: 1.000  LR: 1.00e-03
Epoch [ 17/30]  Loss: 0.3092  Acc: 0.992  LR: 1.00e-03
Epoch [ 18/30]  Loss: 0.3027  Acc: 1.000  LR: 1.00e-03
Epoch [ 19

BERT Solution

In [ ]:
from transformers import BertTokenizer
from tools.bert import IntentClassifier
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

le = LabelEncoder()
y_train  = torch.from_numpy(le.fit_transform(intent_train).astype(np.int64))

encoding = tokenizer(
    text_train.values.tolist(),
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt"
)


model = IntentClassifier(num_classes=len(le.classes_))
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()


X_loader = DataLoader(TensorDataset(encoding['input_ids'], encoding['attention_mask']),
    shuffle=False,
    batch_size=32
)

y_loader = DataLoader(
    y_train,
    shuffle=False,
    batch_size=32
)

model.train()
num_epochs = 10
for epoch in range(num_epochs):
    train_loss = 0
    train_correct = 0
    for (batch_inputs_ids, batch_attention_mask), batch_label in zip(X_loader, y_loader):
        optimizer.zero_grad()
        outputs = model(batch_inputs_ids, batch_attention_mask)

        loss = criterion(outputs, batch_label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_correct += (outputs.argmax(dim=1) == batch_label).sum().item()


    avg_train_loss = train_loss / len(X_loader)
    train_acc      = train_correct / len(X_loader.dataset)
    print(f"Epoch [{epoch+1}/20] Loss: {train_loss/len(X_loader):.4f}")

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.3f}"
    )

test_encoding = tokenizer(
    text_test.values.tolist(),
    padding="max_length",
    truncation=True,
    max_length=32,
    return_tensors="pt"
)
y_test = le.fit_transform(intent_test).astype(np.int64)
preds = None
try:
    with torch.no_grad():
    
        outputs = model(test_encoding['input_ids'], test_encoding['attention_mask'])
        preds   = torch.argmax(outputs, dim=1)
    
    print("Text BERT + FNN accuracy score:",accuracy_score(
            y_true=y_test,
            y_pred=preds.numpy()
        )
    )
except Exception as e:
    print(str(e))
torch.save(model.state_dict(), "models/bert_fnn.pkl")
print("BERT + FNN saved as models/bert_fnn.pkl")




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch [1/20] Loss: 1.9160
Epoch [1/10] Train Loss: 1.9160  Acc: 0.184
Epoch [2/20] Loss: 1.7578
Epoch [2/10] Train Loss: 1.7578  Acc: 0.504
Epoch [3/20] Loss: 1.5515
Epoch [3/10] Train Loss: 1.5515  Acc: 0.650
Epoch [4/20] Loss: 1.3557
Epoch [4/10] Train Loss: 1.3557  Acc: 0.782
Epoch [5/20] Loss: 1.1044
Epoch [5/10] Train Loss: 1.1044  Acc: 0.850
Epoch [6/20] Loss: 0.8711
Epoch [6/10] Train Loss: 0.8711  Acc: 0.921
Epoch [7/20] Loss: 0.6369
Epoch [7/10] Train Loss: 0.6369  Acc: 0.959
Epoch [8/20] Loss: 0.4955
Epoch [8/10] Train Loss: 0.4955  Acc: 0.981


Inference

In [3]:
try:
    with torch.no_grad():
    
        outputs = model(test_encoding['input_ids'], test_encoding['attention_mask'])
        preds   = torch.argmax(outputs, dim=1)
    
    print("Text BERT + FNN accuracy score:", accuracy_score(
            y_true=y_test,
            y_pred=preds.numpy()
        )
    )
except Exception as e:
    print(str(e))

# test_df = pd.read_csv("data/intents_test.csv")

# X_test = processor.transform(test_df['text']).astype(np.float32)

# model = torch.load("models/text_cnn.pkl")
# model.eval()

# X_test = X_test.permute(0, 2, 1)

# preds = None
# with torch.no_grad():
    
#     outputs = model(X_test)
#     preds   = torch.argmax(outputs, dim=1)
    
# df = pd.DataFrame({'intent':preds.numpy().to_list()})
# df.save("intents.csv")

Text CNN accuracy score: 0.9253731343283582
